<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## CMTC: Estado Estable y Colas M/M/1
### T2.3 (parte b) · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Distribución estacionaria: existencia y unicidad
2. Ecuaciones de balance global
3. Balance detallado y reversibilidad
4. Cadenas de nacimiento-muerte
5. Cola M/M/1: derivación y fórmulas de Erlang
6. Sensibilidad a ρ
7. Caso: Metro de Bogotá

## Distribución estacionaria
<div class='defn'>
El vector π∞ = [π∞₀, π∞₁, ..., π∞ₙ₋₁] es la <strong>distribución estacionaria</strong> si:

$$\pi^\infty Q = \mathbf{0}, \qquad \sum_i \pi^\infty_i = 1$$

Bajo condiciones de irreducibilidad y aperiodicidad, π(t) → π∞ cuando t → ∞, independientemente de π(0).
</div>

## Ecuaciones de balance global
La condición πQ = 0 equivale a:

$$\forall j: \quad \pi_j^\infty \, q_j = \sum_{i \neq j} \pi_i^\infty \, q_{ij}$$

**Interpretación (Ley de flujo):**
En estado estacionario, el flujo de probabilidad **saliente** del estado j es igual al flujo **entrante** desde todos los demás estados.

<div class='nota'>
El sistema πQ = 0, ∑πᵢ = 1 tiene siempre una única solución para CMTCs irreducibles de espacio finito. Se resuelve con álgebra lineal reemplazando una ecuación de balance por la restricción de normalización.
</div>

## Balance detallado (reversibilidad)
<div class='defn'>
Una CMTC satisface <strong>balance detallado</strong> si para todo par (i, j):

$$\pi_i^\infty \, q_{ij} = \pi_j^\infty \, q_{ji}$$

El flujo de i a j en estado estacionario iguala el flujo de j a i.
</div>

El balance detallado implica balance global, pero no viceversa.

Las cadenas de **nacimiento-muerte** siempre satisfacen balance detallado (estructura tridiagonal).

## Cadenas de nacimiento-muerte
**Espacio de estados:** {0, 1, 2, ...}
Solo se permiten transiciones entre estados adyacentes:
- Tasa de nacimiento (subida): λᵢ = tasa de i → i+1
- Tasa de muerte (bajada): μᵢ = tasa de i → i-1

**Balance detallado:** π∞ᵢ λᵢ = π∞ᵢ₊₁ μᵢ₊₁

**Solución recursiva:**
$$\pi_n^\infty = \pi_0^\infty \prod_{k=0}^{n-1} \frac{\lambda_k}{\mu_{k+1}}$$

con π∞₀ determinada por la normalización.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from jmarkov.ctmc import ctmc

# ── Sistema de degradación (3 estados) del Tema 2.2 ──────────────────
Q_deg = np.array([[-0.4,  0.3,  0.1],
                  [ 1.0, -1.4,  0.4],
                  [ 0.0,  0.8, -0.8]])
estados_deg = np.array(['Normal', 'Degradado', 'Fallo'])
mc_deg = ctmc(Q_deg, estados_deg)

pi_deg = mc_deg.steady_state()
print("─── Sistema de degradación (T2.2) ───────────────────")
print(f"¿Es ergódica? {mc_deg.is_ergodic()}")
for i, e in enumerate(estados_deg):
    print(f"  π∞({e:10s}) = {pi_deg[i]:.4f}  ({pi_deg[i]*100:.1f}%)")
print(f"\nDisponibilidad (Normal + Degradado) = {(pi_deg[0]+pi_deg[1])*100:.1f}%")

fpt = mc_deg.first_passage_time(2)   # índice 2 = 'Fallo'
print(f"\nE[tiempo hasta primer fallo]:")
for i, e in enumerate(estados_deg[:-1]):
    print(f"  desde {e}: {fpt[i,0]:.2f} h")

# ── Máquina 2 estados (T2.2) ────────────────────────────────────────
Q_mc2 = np.array([[-0.5,  0.5],
                  [ 2.0, -2.0]])
mc2s = ctmc(Q_mc2, np.array(['Operativa', 'Reparación']))
pi2  = mc2s.steady_state()
print(f"\n─── Máquina 2 estados (α=0.5, β=2.0) ────────────────")
print(f"  π∞ = {pi2.round(4)}  →  Disponibilidad = {pi2[0]*100:.1f}%")

## Estado estable con `jmarkov`
Para cualquier CMTC ergódica, `jmarkov` resuelve automáticamente πQ = 0, ∑π = 1:

```python
from jmarkov.ctmc import ctmc
import numpy as np

# Sistema degradación 3 estados (del T2.2)
Q = np.array([[-0.4, 0.3, 0.1],
              [ 1.0, -1.4, 0.4],
              [ 0.0,  0.8, -0.8]])
estados = np.array(['Normal', 'Degradado', 'Fallo'])

mc = ctmc(Q, estados)
pi_inf = mc.steady_state()
# → array([0.5797, 0.2319, 0.1884])
```

<div class='nota'>
Internamente jmarkov reemplaza una ecuación de balance por la restricción de normalización y resuelve el sistema con álgebra lineal numérica estable.
</div>

**Métricas derivadas:**
- `mc.first_passage_time(j)` — E[tiempo hasta llegar por primera vez al estado j]
- `mc.occupation_time(T)` — E[horas en cada estado hasta el horizonte T]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from jmarkov.ctmc import ctmc

# Cola M/M/1: lambda_i = lambda, mu_i = mu para todo i > 0
lam, mu = 3.0, 5.0
rho = lam / mu
N_max = 20   # estados para graficar

# ── Distribución estacionaria exacta (fórmula geométrica) ────────────
pi = np.array([(1 - rho) * rho**n for n in range(N_max + 1)])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(N_max + 1), pi, color='#1A7A9A', edgecolor='white', alpha=0.85)
ax.set_xlabel('Número de clientes en el sistema (n)')
ax.set_ylabel('π∞ₙ')
ax.set_title(f'Distribución estacionaria M/M/1  (λ={lam}, μ={mu}, ρ={rho:.2f})')

# Métricas analíticas
L  = rho / (1 - rho)
Lq = rho**2 / (1 - rho)
W  = 1 / (mu - lam)
Wq = rho / (mu - lam)
ax.text(0.62, 0.72,
        f"L  = {L:.3f}\nLq = {Lq:.3f}\nW  = {W:.3f}\nWq = {Wq:.3f}",
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', fc='#EAF6FA', ec='#1A7A9A'))
plt.tight_layout(); plt.show()

# ── Verificación con jmarkov (Q matricial, cadena truncada) ──────────
N_tr = 30   # suficiente: π(30) ≈ 2e-7 para ρ=0.6
Q_mm1 = np.zeros((N_tr + 1, N_tr + 1))
for i in range(N_tr):
    Q_mm1[i, i+1] = lam          # llegada
    Q_mm1[i+1, i] = mu           # servicio
np.fill_diagonal(Q_mm1, -Q_mm1.sum(axis=1))

mc_mm1 = ctmc(Q_mm1, np.arange(N_tr + 1))
pi_jm  = mc_mm1.steady_state()

print("Estado n | Fórmula exacta | jmarkov | Error")
print("-" * 48)
for n in range(8):
    err = abs(pi[n] - pi_jm[n])
    print(f"  n={n}     |   {pi[n]:.6f}    | {pi_jm[n]:.6f} | {err:.2e}")

L_jm = sum(n * pi_jm[n] for n in range(N_tr + 1))
print(f"\nL analítico = {L:.4f}  |  L jmarkov = {L_jm:.4f}")

## Fórmulas de Erlang — Cola M/M/1
Para λ < μ (condición de estabilidad ρ < 1):

| Métrica | Fórmula | Descripción |
|---|---|---|
| **ρ** | λ/μ | Utilización del servidor |
| **L** | ρ/(1−ρ) | Clientes promedio en el sistema |
| **Lq** | ρ²/(1−ρ) | Clientes promedio en la cola |
| **W** | 1/(μ−λ) | Tiempo promedio en el sistema |
| **Wq** | ρ/(μ−λ) | Tiempo promedio en la cola |
| **P(n)** | (1−ρ)ρⁿ | P(exactamente n clientes) |

**Ley de Little:** L = λW, Lq = λWq

In [ ]:
# Sensibilidad a ρ
rhos = np.linspace(0.01, 0.99, 300)
L_vals  = rhos / (1 - rhos)
Lq_vals = rhos**2 / (1 - rhos)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rhos, L_vals,  color='#1A7A9A', lw=2.5, label='L  (en sistema)')
axes[0].plot(rhos, Lq_vals, color='#C8961E', lw=2.5, label='Lq (en cola)')
axes[0].axvline(0.9, ls='--', color='#C0392B', lw=1.5, label='ρ = 0.9')
axes[0].set_xlabel('Utilización ρ = λ/μ')
axes[0].set_ylabel('Número promedio de clientes')
axes[0].set_title('M/M/1: L y Lq en función de ρ')
axes[0].set_ylim(0, 20); axes[0].legend()

# W como función de ρ (fijando μ = 1)
mu_fixed = 1.0
W_vals = 1 / (mu_fixed - rhos * mu_fixed)
axes[1].plot(rhos, W_vals, color='#1A7A9A', lw=2.5)
axes[1].axvline(0.9, ls='--', color='#C0392B', lw=1.5, label='ρ = 0.9')
axes[1].set_xlabel('Utilización ρ')
axes[1].set_ylabel('W (tiempo promedio en sistema)')
axes[1].set_title('M/M/1: W en función de ρ  (μ=1)')
axes[1].set_ylim(0, 20); axes[1].legend()
plt.tight_layout(); plt.show()

## Caso: Metro de Bogotá
**Situación:** Pasajeros llegan a una estación de Transmilenio/Metro siguiendo un PP(λ).
El sistema de torniquetes procesa a tasa μ (1 servidor efectivo en hora pico).

**Datos observados (hora pico):**
- Llegadas: ~360 personas/hora → λ = 6 por minuto
- Tiempo de validación promedio: 12 segundos → μ = 5 por minuto

**Pregunta:** ¿Es estable el sistema? ¿Cuánto espera un pasajero en promedio?

In [ ]:
# ── Metro de Bogotá — modelo M/M/c con jmarkov ───────────────────────
lam_metro = 6.0    # pasajeros/minuto
mu_metro  = 5.0    # validaciones/minuto por torniquete

print("=" * 55)
print("ESCENARIO 1: 1 torniquete (M/M/1)")
print("=" * 55)
rho1 = lam_metro / mu_metro
print(f"ρ = {rho1:.2f}  → {'INESTABLE ⚠' if rho1 >= 1 else 'estable'}")

print("\n" + "=" * 55)
print("ESCENARIO 2: 2 torniquetes (M/M/2)")
print("=" * 55)
c = 2
N_tr = 30   # truncar

# Construir Q para M/M/c (nacimiento-muerte)
Q_mmc = np.zeros((N_tr + 1, N_tr + 1))
for n in range(N_tr):
    Q_mmc[n, n+1] = lam_metro                          # llegada
    Q_mmc[n+1, n] = min(n+1, c) * mu_metro             # servicio (c servidores)
np.fill_diagonal(Q_mmc, -Q_mmc.sum(axis=1))

mc_mmc = ctmc(Q_mmc, np.arange(N_tr + 1))
pi_mmc = mc_mmc.steady_state()
rho_c  = lam_metro / (c * mu_metro)

L_mmc  = sum(n * pi_mmc[n] for n in range(N_tr + 1))
Lq_mmc = sum(max(n - c, 0) * pi_mmc[n] for n in range(N_tr + 1))
W_mmc  = L_mmc / lam_metro
Wq_mmc = Lq_mmc / lam_metro

print(f"ρ_efectiva = λ/(cμ) = {rho_c:.2f}  → {'INESTABLE ⚠' if rho_c >= 1 else 'estable ✓'}")
print(f"\nResultados [jmarkov]:")
print(f"  L  = {L_mmc:.3f}  pasajeros en sistema")
print(f"  Lq = {Lq_mmc:.3f}  pasajeros en cola")
print(f"  W  = {W_mmc:.3f}  min en sistema")
print(f"  Wq = {Wq_mmc:.3f}  min en cola")

# Comparación visual: tiempo en cola vs número de torniquetes
fig, ax = plt.subplots(figsize=(9, 4))
torniquetes = [2, 3, 4, 5]
Wq_vals = []
for c_i in torniquetes:
    Q_i = np.zeros((N_tr+1, N_tr+1))
    for n in range(N_tr):
        Q_i[n, n+1] = lam_metro
        Q_i[n+1, n] = min(n+1, c_i) * mu_metro
    np.fill_diagonal(Q_i, -Q_i.sum(axis=1))
    pi_i = ctmc(Q_i, np.arange(N_tr+1)).steady_state()
    Lq_i = sum(max(n-c_i,0)*pi_i[n] for n in range(N_tr+1))
    Wq_vals.append(Lq_i / lam_metro)

ax.bar(torniquetes, Wq_vals, color='#1A7A9A', edgecolor='white', alpha=0.85)
ax.set_xlabel('Número de torniquetes (c)'); ax.set_ylabel('Tiempo en cola Wq (min)')
ax.set_title('Metro Bogotá — impacto del número de torniquetes')
for x, y in zip(torniquetes, Wq_vals):
    ax.text(x, y + 0.005, f'{y:.3f}', ha='center', fontsize=10)
plt.tight_layout(); plt.show()

## Conclusiones
- La distribución estacionaria π∞ satisface π∞Q = 0, ∑π∞ᵢ = 1.
- El balance detallado (queueing reversibility) simplifica la solución en cadenas B-M.
- La cola M/M/1 tiene solución geométrica exacta con parámetro ρ = λ/μ.
- El sistema es **solo estable si ρ < 1**; para ρ cercano a 1 los tiempos de espera crecen drásticamente.
- La Ley de Little (L = λW) conecta número en sistema con tiempo de permanencia.

**Módulo 3:** Modelado de entradas, V&V y análisis estadístico de salida de simulación.